In [1]:
import pyodbc
import pandas as pd 
import numpy as np

In [2]:
import pyodbc
import pandas as pd
DRIVER_NAME = 'ODBC Driver 17 for SQL Server'
SERVER_NAME = r'DESKTOP-L3GBMQ5\SQLEXPRESS'
DATABASE_NAME = 'Financial_CaseStudy'

connection_string = (
    f"DRIVER={{{DRIVER_NAME}}};"
    f"SERVER={SERVER_NAME};"
    f"DATABASE={DATABASE_NAME};"
    f"Trusted_Connection=yes;"
)

conn = pyodbc.connect(connection_string)
cursor = conn.cursor()
print("Connected successfully!")

Connected successfully!


------------------------------------------------------------------------------------------------------------------

### Lod Staging Tables

In [3]:
tables = ['stg.stg_dim_customers', 'stg.stg_fact_payments', 'stg.stg_loan_applications']
dfs={}

In [6]:
for table in tables:
    query = f"SELECT * FROM {table};"
    print(f" Loading {table} ...")
    df = pd.read_sql(query, conn)
    dfs[table] = df
    print(f"   {len(df)} rows loaded.")

 Loading stg.stg_dim_customers ...


C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_9096\928251109.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


   500 rows loaded.
 Loading stg.stg_fact_payments ...
   3000 rows loaded.
 Loading stg.stg_loan_applications ...
   1000 rows loaded.


--------------------------------------------

### Loan Applications Cleaning

In [7]:
loan_df = dfs['stg.stg_loan_applications'].copy()

In [11]:
loan_df

,Loan_ID,Customer_ID,Loan_Amount,Loan_Term_Months,Interest_Rate,Loan_Purpose,Application_Date,Approval_Status,Default_Status,Monthly_Income,Credit_Score,Employment_Length,Marital_Status,Age,Region,Gender
0,5001,1059,48370.0,24,11.91,Education,2024-11-11,Approved,Paid,10828.833008,391,18,Married,24,North,Male
1,5002,1132,43396.0,36,10.15,Car,2023-01-05,Rejected,None,2990.500000,696,19,Divorced,39,East,Male
2,5003,1104,13328.0,24,12.97,Business,2022-07-10,Approved,Paid,2570.416748,754,3,Single,9,Central,Female
3,5004,1195,20534.0,36,15.47,Education,2022-12-13,Approved,Paid,10096.500000,677,9,Single,45,West,Male
4,5005,1413,24744.0,12,8.90,Business,2022-09-10,Approved,Paid,6738.333496,789,4,Married,5,Central,Female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,5996,1147,6881.0,36,6.81,Business,2022-09-28,Approved,Paid,3032.500000,499,6,Divorced,37,Central,Male
996,5997,1118,44567.0,36,11.76,Business,2021-05-30,Approved,Defaulted,8870.166992,637,13,Married,5,South,Female
997,5998,1399,18675.0,12,5.48,Personal,2021-08-03,Approved,Paid,7271.750000,441,7,Single,31,South,Female
998,5999,1382,9586.0,24,6.05,Car,2022-12-17,Approved,Defaulted,11057.666992,699,5,Divorced,10,North,Female


In [9]:
# Remove duplicates Loan_ID
loan_df = loan_df.drop_duplicates(subset = 'Loan_ID', keep = 'first')

In [12]:
# Remove records without Customer_ID or Loan_Amount
loan_df = loan_df.dropna(subset=['Customer_ID', 'Loan_Amount'])

In [13]:
# Replace negative or zero values
loan_df.loc[loan_df['Loan_Term_Months'] <= 0, 'Loan_Term_Months'] = np.nan
loan_df.loc[loan_df['Loan_Amount'] <= 0, 'Loan_Amount'] = np.nan

In [14]:
# Fill missing Interest_Rate with mean
if 'Interest_Rate' in loan_df.columns:
    loan_df['Interest_Rate'] = loan_df['Interest_Rate'].fillna(loan_df['Interest_Rate'].mean())

In [15]:
# Fill missing Application_Date with earliest available date
if 'Application_Date' in loan_df.columns:
    min_date = loan_df['Application_Date'].min()
    loan_df['Application_Date'] = loan_df['Application_Date'].fillna(min_date)

In [16]:
print(f" Loan Applications cleaned → {len(loan_df)} rows remain.")

 Loan Applications cleaned → 1000 rows remain.


--------------------------------------

### Customers Cleaning

In [17]:
cust_df = dfs['stg.stg_dim_customers'].copy()

In [19]:
cust_df

,Customer_ID,Full_Name,Gender,Date_of_Birth,Region,Education_Level,Employment_Status,Annual_Income,Credit_History_Length,Credit_Score
0,1001,Customer_1,Male,1996-07-22,North,High School,Employed,102988.0,3,808
1,1002,Customer_2,Female,2012-01-23,North,Bachelor,Retired,83342.0,1,780
2,1003,Customer_3,Male,1986-08-12,West,Master,Self-employed,139516.0,4,413
3,1004,Customer_4,Male,1996-05-18,West,Bachelor,Employed,43951.0,20,660
4,1005,Customer_5,Male,1997-11-21,West,High School,Employed,26307.0,2,640
...,...,...,...,...,...,...,...,...,...,...
495,1496,Customer_496,Male,1988-12-04,North,Master,Self-employed,72832.0,18,559
496,1497,Customer_497,Male,2000-07-24,Central,Bachelor,Employed,53151.0,8,401
497,1498,Customer_498,Female,1989-09-08,West,Bachelor,Employed,104303.0,22,805
498,1499,Customer_499,Male,2013-08-17,West,Master,Retired,36308.0,24,546


In [18]:
cust_df = cust_df.drop_duplicates(subset = 'Customer_ID', keep = 'first')

In [20]:
cust_df = cust_df.dropna(subset=['Customer_ID', 'Full_Name'])

In [21]:
# Fill missingg Income with median
if 'Annual_Income' in cust_df.columns:
    cust_df['Annual_Income'] = cust_df['Annual_Income'].fillna(cust_df['Annual_Income'].median())

In [22]:
# Standardize Gender values
if 'Gender' in cust_df.columns:
    cust_df['Gender'] = cust_df['Gender'].str.upper().replace({
        'M': 'Male', "MALE" : 'Male',
        'F': 'Female', "FEMALE": "Female"
    })

In [23]:
print(f" Customers cleaned : {len(cust_df)} rows remain.")


 Customers cleaned : 500 rows remain.


----------------

### Clean Payments

In [24]:
pay_df = dfs['stg.stg_fact_payments'].copy()

In [25]:
pay_df

,Payment_ID,Loan_ID,Payment_Date,Payment_Amount,Remaining_Balance,Is_Late,Days_Late
0,8001,5447,2021-05-01,645.760010,26237.949219,False,0
1,8002,5785,2021-12-11,743.880005,4793.629883,True,12
2,8003,5497,2020-10-08,1407.630005,35170.910156,False,0
3,8004,5561,2020-07-25,1383.979980,31168.130859,False,0
4,8005,5484,2023-10-12,1130.380005,7629.930176,False,0
...,...,...,...,...,...,...,...
2995,10996,5650,2021-08-21,1355.359985,39588.929688,False,0
2996,10997,5983,2022-11-10,803.679993,19366.570312,False,0
2997,10998,5466,2022-08-08,870.960022,18723.259766,False,0
2998,10999,5288,2023-07-05,632.590027,25798.550781,True,18


In [26]:
pay_df = pay_df.drop_duplicates(subset = 'Payment_ID', keep = 'first')

In [29]:
pay_df = pay_df.dropna(subset=['Payment_ID', 'Loan_ID', 'Payment_Amount', 'Payment_Date'])

In [30]:
# Repalce negative payments with NULL
pay_df.loc[pay_df['Payment_Amount'] <= 0, 'Payment_Amount'] = np.nan

In [31]:
print(f" Payments cleaned : {len(pay_df)} rows remain.")


 Payments cleaned : 3000 rows remain.


------------------------------